In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from pandas.core.arrays import categorical
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from xgboost import XGBRegressor
from keras.layers import Input
from google.colab import files
import zipfile

In [ ]:
uploaded = files.upload()
with zipfile.ZipFile("digit-recognizer.zip","r")as zip_ref:
  zip_ref.extractall()

Saving digit-recognizer.zip to digit-recognizer (2).zip


In [ ]:
train=pd.read_csv("train.csv")
test=pd.read_csv("test.csv")

In [ ]:
print(train.describe())
print(train.info())
print(train.shape)
print(train.columns)
print(train.head())
train.isna().sum()

              label   pixel0   pixel1   pixel2   pixel3   pixel4   pixel5  \
count  42000.000000  42000.0  42000.0  42000.0  42000.0  42000.0  42000.0   
mean       4.456643      0.0      0.0      0.0      0.0      0.0      0.0   
std        2.887730      0.0      0.0      0.0      0.0      0.0      0.0   
min        0.000000      0.0      0.0      0.0      0.0      0.0      0.0   
25%        2.000000      0.0      0.0      0.0      0.0      0.0      0.0   
50%        4.000000      0.0      0.0      0.0      0.0      0.0      0.0   
75%        7.000000      0.0      0.0      0.0      0.0      0.0      0.0   
max        9.000000      0.0      0.0      0.0      0.0      0.0      0.0   

        pixel6   pixel7   pixel8  ...      pixel774      pixel775  \
count  42000.0  42000.0  42000.0  ...  42000.000000  42000.000000   
mean       0.0      0.0      0.0  ...      0.219286      0.117095   
std        0.0      0.0      0.0  ...      6.312890      4.633819   
min        0.0      0.0      0

,0
label,0
pixel0,0
pixel1,0
pixel2,0
pixel3,0
...,...
pixel779,0
pixel780,0
pixel781,0
pixel782,0


In [ ]:
X_train=train.drop("label", axis=1)
Y_train=train["label"].copy()
X_test=test.copy()

In [ ]:
SV_model=SVC()
SV_model.fit(X_train,Y_train)
pred=SV_model.predict(X_test)

In [ ]:
X_train = X_train / 255.0
X_test = X_test / 255.0
X_train=X_train.to_numpy().reshape(-1, 28, 28, 1)
X_test=X_test.to_numpy().reshape(-1, 28, 28, 1)

In [ ]:
model=Sequential([
    Input(shape=(28,28,1)),
    Conv2D(32,(3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
    ])


In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])
model.fit(
    X_train,
    Y_train,
    epochs=3,
    validation_split=0.1)
# Conv2D (32, (3,3)): دي "العيون". بتعدي بـ 32 "كشاف" (Filters) على الصورة عشان تلقط الأنماط (خطوط، دوائر، زوايا).

# MaxPooling2D (2,2): دي "المختصراتي". بتقلل حجم الصورة للنص عشان تركز بس على أهم بكسلات (الأقوى) وتوفر في الحسابات.

# Flatten(): الطبقات اللي فوق كانت بتشوف الصورة "مربعة"، الـ Dense اللي تحتها بتفهم "خطوط طولية". السطر ده بيفرد المربع ويخليه خط واحد طويل من الأرقام.

# Dense (64): دي طبقة التفكير العميق، بتربط كل المعلومات ببعضها.

# Dense (10, activation='softmax'): دي "محطة القرار". رقم 10 معناه إنك بتصنفي 10 أنواع (أرقام من 0 لـ 9 مثلاً). الـ Softmax بتحول النتائج لـ احتمالات (مثلاً: 90% احتمال إن ده رقم 5).

Epoch 1/3
1182/1182 ━━━━━━━━━━━━━━━━━━━━ 24s 20ms/step - accuracy: 0.8722 - loss: 0.4246 - val_accuracy: 0.9736 - val_loss: 0.0887
Epoch 2/3
1182/1182 ━━━━━━━━━━━━━━━━━━━━ 23s 19ms/step - accuracy: 0.9773 - loss: 0.0755 - val_accuracy: 0.9793 - val_loss: 0.0636
Epoch 3/3
1182/1182 ━━━━━━━━━━━━━━━━━━━━ 25s 21ms/step - accuracy: 0.9852 - loss: 0.0485 - val_accuracy: 0.9810 - val_loss: 0.0593


In [ ]:
predictions=model.predict(X_test)
pred_labels = predictions.argmax(axis=1)

875/875 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step


In [ ]:
submission = pd.DataFrame({
    "ImageId": range(1, len(pred_labels) + 1),
    "Label":pred_labels })
submission.to_csv("submission.csv", index=False)
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>